In [1]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error



# Histogram binning helper

def create_bins(X, num_bins=255):
    """Create bin edges for each feature using quantiles."""
    n_features = X.shape[1]
    bin_edges = []
    for f in range(n_features):
        col = X[:, f]
        # Get quantile-based bins (ignore NaNs if any)
        quantiles = np.linspace(0, 100, num_bins + 1)
        edges = np.percentile(col, quantiles)
        # Ensure unique edges (remove duplicates)
        edges = np.unique(edges)
        bin_edges.append(edges)
    return bin_edges

def discretize(X, bin_edges):
    """Convert continuous features to bin indices (0..num_bins-1)."""
    n_samples = X.shape[0]
    n_features = X.shape[1]
    X_binned = np.zeros((n_samples, n_features), dtype=np.int32)
    for f in range(n_features):
        X_binned[:, f] = np.digitize(X[:, f], bin_edges[f][:-1]) - 1
        # Ensure values are within [0, len(bin_edges[f])-2]
        max_idx = len(bin_edges[f]) - 2
        X_binned[:, f] = np.clip(X_binned[:, f], 0, max_idx)
    return X_binned



# Histogram-based Decision Tree (leaf-wise growth)

class LightGBMTree:
    """
    Decision tree built using histograms and leaf-wise growth.
    """
    def __init__(self, num_leaves=31, min_data_in_leaf=20, min_child_weight=1e-3,
                 lambda_=1.0, gamma=0.0, max_depth=-1, feature_fraction=1.0):
        self.num_leaves = num_leaves
        self.min_data_in_leaf = min_data_in_leaf
        self.min_child_weight = min_child_weight
        self.lambda_ = lambda_
        self.gamma = gamma
        self.max_depth = max_depth
        self.feature_fraction = feature_fraction
        self.bin_edges = None
        self.X_binned = None

    def _compute_gain(self, G, H, G_L, H_L, G_R, H_R):
        """Gain = 0.5*(G_L^2/(H_L+λ) + G_R^2/(H_R+λ) - (G_L+G_R)^2/(H_L+H_R+λ)) - γ"""
        def term(g, h):
            return g**2 / (h + self.lambda_)
        gain = 0.5 * (term(G_L, H_L) + term(G_R, H_R) - term(G_L + G_R, H_L + H_R)) - self.gamma
        return gain

    def _build_histograms(self, X_binned, G, H, feature_indices):
        """
        For each feature in feature_indices, build histograms of G and H sums per bin.
        Returns: dict feature -> (hist_G, hist_H, bin_edges)
        """
        n_samples = X_binned.shape[0]
        n_bins_per_feature = [len(edges) - 1 for edges in self.bin_edges]
        histograms = {}
        for f in feature_indices:
            n_bins = n_bins_per_feature[f]
            hist_G = np.zeros(n_bins, dtype=np.float64)
            hist_H = np.zeros(n_bins, dtype=np.float64)
            # Use np.add.at for efficiency
            bin_ids = X_binned[:, f]
            np.add.at(hist_G, bin_ids, G)
            np.add.at(hist_H, bin_ids, H)
            histograms[f] = (hist_G, hist_H)
        return histograms

    def _find_best_split(self, node_samples, G, H, X_binned, histograms):
        """
        Given the samples in a node, find the best split across all features.
        Returns: (best_feature, best_bin, left_mask, right_mask, best_gain)
        """
        n_samples = len(node_samples)
        if n_samples < 2 * self.min_data_in_leaf:
            return None, None, None, None, -np.inf

        total_G = np.sum(G[node_samples])
        total_H = np.sum(H[node_samples])
        if total_H < self.min_child_weight:
            return None, None, None, None, -np.inf

        best_gain = -np.inf
        best_feature = None
        best_bin = None
        best_left_mask = None
        best_right_mask = None

        # Only consider a subset of features (feature_fraction)
        features = list(histograms.keys())
        if self.feature_fraction < 1.0:
            n_sub = max(1, int(len(features) * self.feature_fraction))
            features = np.random.choice(features, n_sub, replace=False)

        for f in features:
            hist_G, hist_H = histograms[f]
            # For each bin, compute left (<=bin) and right (>bin) sums
            cum_G = 0.0
            cum_H = 0.0
            # We need to iterate over bins, but we can use cumulative sums
            # But careful: we need to consider bins that have samples
            # We'll loop over bins and accumulate
            for b in range(len(hist_G)):
                # Include bin b in left
                if hist_H[b] > 0:
                    cum_G += hist_G[b]
                    cum_H += hist_H[b]
                    # If left/right have enough data
                    if cum_H < self.min_child_weight or (total_H - cum_H) < self.min_child_weight:
                        continue
                    # Gain
                    gain = self._compute_gain(total_G, total_H, cum_G, cum_H, total_G - cum_G, total_H - cum_H)
                    if gain > best_gain:
                        best_gain = gain
                        best_feature = f
                        best_bin = b
                        # Build masks for this split
                        left_mask = (X_binned[node_samples, f] <= b)
                        right_mask = ~left_mask
                        best_left_mask = left_mask
                        best_right_mask = right_mask
        return best_feature, best_bin, best_left_mask, best_right_mask, best_gain

    def _grow_leaf_wise(self, X_binned, G, H, node_samples):
        """Grow tree using leaf-wise strategy."""
        # Node structure: each node is a dict with 'samples', 'depth', 'left', 'right', 'feature', 'bin', 'weight'
        # We'll use lists and indices
        nodes = []
        # First node (root)
        root = {
            'id': 0,
            'samples': node_samples,
            'depth': 0,
            'left': -1,
            'right': -1,
            'feature': -1,
            'bin': -1,
            'is_leaf': False,
            'weight': self._compute_leaf_weight(G[node_samples], H[node_samples])
        }
        nodes.append(root)

        leaves = [0]  # list of node indices that are leaves (candidate for splitting)
        n_leaves = 1

        while len(leaves) > 0 and n_leaves < self.num_leaves:
            # Find leaf with best gain (largest gain)
            best_leaf_idx = None
            best_gain = -np.inf
            best_split_info = None

            for leaf_id in leaves:
                node = nodes[leaf_id]
                if node['is_leaf']:
                    continue
                # Build histograms for this leaf's samples
                samples = node['samples']
                if len(samples) < 2 * self.min_data_in_leaf:
                    continue
                # Compute histograms (using all features)
                # We'll build histograms for all features using the node samples
                # For speed, we compute histograms on the fly for each leaf
                # In a full implementation we'd cache, but for clarity we recompute
                # We'll create histograms for all features
                n_features = X_binned.shape[1]
                n_bins_per_feature = [len(edges) - 1 for edges in self.bin_edges]
                histograms = {}
                for f in range(n_features):
                    hist_G = np.zeros(n_bins_per_feature[f], dtype=np.float64)
                    hist_H = np.zeros(n_bins_per_feature[f], dtype=np.float64)
                    bin_ids = X_binned[samples, f]
                    G_sub = G[samples]
                    H_sub = H[samples]
                    np.add.at(hist_G, bin_ids, G_sub)
                    np.add.at(hist_H, bin_ids, H_sub)
                    histograms[f] = (hist_G, hist_H)

                # Find best split
                best_f, best_b, left_mask, right_mask, gain = self._find_best_split(
                    samples, G, H, X_binned, histograms)
                if gain > best_gain:
                    best_gain = gain
                    best_leaf_idx = leaf_id
                    best_split_info = (best_f, best_b, left_mask, right_mask, gain)

            if best_leaf_idx is None or best_gain <= 0:
                # No more beneficial splits
                break

            # Apply best split to the best leaf
            leaf_node = nodes[best_leaf_idx]
            f, b, left_mask, right_mask, gain = best_split_info
            # Create left and right child nodes
            left_samples = leaf_node['samples'][left_mask]
            right_samples = leaf_node['samples'][right_mask]
            if len(left_samples) < self.min_data_in_leaf or len(right_samples) < self.min_data_in_leaf:
                # Should not happen due to checks, but safety
                leaf_node['is_leaf'] = True
                leaves.remove(best_leaf_idx)
                continue

            # Create left child
            left_child = {
                'id': len(nodes),
                'samples': left_samples,
                'depth': leaf_node['depth'] + 1,
                'left': -1,
                'right': -1,
                'feature': -1,
                'bin': -1,
                'is_leaf': False,
                'weight': self._compute_leaf_weight(G[left_samples], H[left_samples])
            }
            nodes.append(left_child)
            left_child_id = len(nodes) - 1

            # Create right child
            right_child = {
                'id': len(nodes),
                'samples': right_samples,
                'depth': leaf_node['depth'] + 1,
                'left': -1,
                'right': -1,
                'feature': -1,
                'bin': -1,
                'is_leaf': False,
                'weight': self._compute_leaf_weight(G[right_samples], H[right_samples])
            }
            nodes.append(right_child)
            right_child_id = len(nodes) - 1

            # Update current leaf node to be internal
            leaf_node['feature'] = f
            leaf_node['bin'] = b
            leaf_node['left'] = left_child_id
            leaf_node['right'] = right_child_id
            leaf_node['is_leaf'] = False
            leaf_node['weight'] = 0.0  # internal nodes have no weight

            # Remove current leaf from leaves list, add new leaves (if depth not exceeded)
            leaves.remove(best_leaf_idx)
            if self.max_depth == -1 or left_child['depth'] < self.max_depth:
                leaves.append(left_child_id)
            else:
                left_child['is_leaf'] = True
            if self.max_depth == -1 or right_child['depth'] < self.max_depth:
                leaves.append(right_child_id)
            else:
                right_child['is_leaf'] = True

            n_leaves += 1  # we increased leaves by 1 (removed one, added two)

        # Set any remaining leaves as leaf nodes
        for leaf_id in leaves:
            nodes[leaf_id]['is_leaf'] = True

        self.tree_nodes = nodes
        return nodes

    def _compute_leaf_weight(self, G, H):
        return -np.sum(G) / (np.sum(H) + self.lambda_)

    def fit(self, X, G, H, bin_edges=None):
        """
        Fit the tree to gradients and Hessians.
        X: original features (will be binned using bin_edges)
        G, H: arrays of gradient and hessian per sample
        bin_edges: precomputed bin edges; if None, compute from X.
        """
        self.bin_edges = bin_edges if bin_edges is not None else create_bins(X)
        self.X_binned = discretize(X, self.bin_edges)

        # All samples in root
        n_samples = X.shape[0]
        node_samples = np.arange(n_samples, dtype=np.int64)
        self.tree_nodes = self._grow_leaf_wise(self.X_binned, G, H, node_samples)
        return self

    def predict(self, X):
        """
        Predict leaf weights for each sample.
        """
        # Bin X
        X_binned = discretize(X, self.bin_edges)
        n_samples = X.shape[0]
        preds = np.zeros(n_samples)

        # For each sample, traverse tree to leaf
        for i in range(n_samples):
            node_id = 0
            while True:
                node = self.tree_nodes[node_id]
                if node['is_leaf']:
                    preds[i] = node['weight']
                    break
                # else split
                if X_binned[i, node['feature']] <= node['bin']:
                    node_id = node['left']
                else:
                    node_id = node['right']
        return preds



# LightGBM Booster (gradient boosting with histogram-based trees)

class LightGBMBase:
    def __init__(self, n_estimators=100, learning_rate=0.1,
                 num_leaves=31, min_data_in_leaf=20, min_child_weight=1e-3,
                 lambda_=1.0, gamma=0.0, max_depth=-1, feature_fraction=1.0,
                 subsample=1.0, early_stopping_rounds=None, random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.min_data_in_leaf = min_data_in_leaf
        self.min_child_weight = min_child_weight
        self.lambda_ = lambda_
        self.gamma = gamma
        self.max_depth = max_depth
        self.feature_fraction = feature_fraction
        self.subsample = subsample
        self.early_stopping_rounds = early_stopping_rounds
        self.random_state = random_state
        self.trees = []
        self.initial_pred = None
        self.bin_edges = None

    def _init_prediction(self, y):
        raise NotImplementedError

    def _get_gradient_hessian(self, y, pred):
        raise NotImplementedError

    def fit(self, X, y, X_val=None, y_val=None):
        X, y = check_X_y(X, y)
        np.random.seed(self.random_state)
        self.trees = []
        n_samples = X.shape[0]

        # Compute bin edges from training data (once)
        self.bin_edges = create_bins(X)

        # Initial prediction
        self.initial_pred = self._init_prediction(y)
        pred = np.full(n_samples, self.initial_pred)

        # Validation predictions if provided
        if X_val is not None:
            y_val = np.array(y_val)
            pred_val = np.full(X_val.shape[0], self.initial_pred)

        best_val_loss = np.inf
        no_improve = 0

        for i in range(self.n_estimators):
            # Subsample
            if self.subsample < 1.0:
                idx = np.random.choice(n_samples, int(n_samples * self.subsample), replace=False)
                X_sub = X[idx]
                y_sub = y[idx]
                pred_sub = pred[idx]
            else:
                X_sub = X
                y_sub = y
                pred_sub = pred

            # Compute gradients and Hessians on subsample
            G, H = self._get_gradient_hessian(y_sub, pred_sub)
            # Fit a tree
            tree = LightGBMTree(
                num_leaves=self.num_leaves,
                min_data_in_leaf=self.min_data_in_leaf,
                min_child_weight=self.min_child_weight,
                lambda_=self.lambda_,
                gamma=self.gamma,
                max_depth=self.max_depth,
                feature_fraction=self.feature_fraction
            )
            tree.fit(X_sub, G, H, bin_edges=self.bin_edges)

            # Update predictions
            pred += self.learning_rate * tree.predict(X)
            self.trees.append(tree)

            # Early stopping on validation
            if X_val is not None:
                pred_val += self.learning_rate * tree.predict(X_val)
                val_loss = self._loss(y_val, pred_val)
                if val_loss < best_val_loss - 1e-7:
                    best_val_loss = val_loss
                    no_improve = 0
                else:
                    no_improve += 1
                if self.early_stopping_rounds is not None and no_improve >= self.early_stopping_rounds:
                    # Keep only the best trees
                    self.trees = self.trees[:i+1 - self.early_stopping_rounds]
                    break

        return self

    def _loss(self, y, pred):
        raise NotImplementedError

    def predict_raw(self, X):
        X = check_array(X)
        pred = np.full(X.shape[0], self.initial_pred)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)
        return pred



# Regressor and Classifier

class LightGBMRegressor(LightGBMBase, RegressorMixin):
    """LightGBM for regression (squared error)."""
    def _init_prediction(self, y):
        return np.mean(y)

    def _get_gradient_hessian(self, y, pred):
        G = pred - y
        H = np.ones_like(y)
        return G, H

    def _loss(self, y, pred):
        return np.mean((y - pred) ** 2)

    def predict(self, X):
        return self.predict_raw(X)


class LightGBMClassifier(LightGBMBase, ClassifierMixin):
    """LightGBM for binary classification (log loss)."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.classes_ = None

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-x))

    def _init_prediction(self, y):
        self.classes_ = np.unique(y)
        p_pos = np.mean(y == self.classes_[1])
        p_pos = np.clip(p_pos, 1e-15, 1.0 - 1e-15)
        return np.log(p_pos / (1.0 - p_pos))

    def _get_gradient_hessian(self, y, pred):
        # y: 0/1
        p = self._sigmoid(pred)
        G = p - y
        H = p * (1.0 - p)
        return G, H

    def _loss(self, y, pred):
        p = self._sigmoid(pred)
        p = np.clip(p, 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def predict_proba(self, X):
        raw = self.predict_raw(X)
        p_pos = self._sigmoid(raw)
        p_pos = np.clip(p_pos, 1e-15, 1 - 1e-15)
        return np.column_stack((1.0 - p_pos, p_pos))

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.where(proba[:, 1] >= 0.5, self.classes_[1], self.classes_[0])



# Example usage

if __name__ == "__main__":
    # ----- Regression -----
    print("--- Regression ---")
    X, y = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    lgb_reg = LightGBMRegressor(
        n_estimators=100, learning_rate=0.1, num_leaves=15,
        min_data_in_leaf=5, lambda_=0.1, gamma=0.0, max_depth=6,
        feature_fraction=0.8, subsample=0.8, random_state=42
    )
    lgb_reg.fit(X_train, y_train)
    y_pred = lgb_reg.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Test MSE: {mse:.3f}")

    # ----- Binary Classification -----
    print("\n--- Classification ---")
    X, y = make_classification(n_samples=500, n_features=10, n_informative=8,
                               n_redundant=2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    lgb_clf = LightGBMClassifier(
        n_estimators=100, learning_rate=0.1, num_leaves=15,
        min_data_in_leaf=5, lambda_=0.1, gamma=0.0, max_depth=6,
        feature_fraction=0.8, subsample=0.8, random_state=42,
        early_stopping_rounds=10
    )
    # Use validation set for early stopping
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    lgb_clf.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    y_pred = lgb_clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test accuracy: {acc:.3f}")

    # Probabilities for first 5
    proba = lgb_clf.predict_proba(X_test[:5])
    print("Predicted probabilities (first 5):\n", proba)

--- Regression ---
Test MSE: 3253.782

--- Classification ---
Test accuracy: 0.847
Predicted probabilities (first 5):
 [[0.51588973 0.48411027]
 [0.18987332 0.81012668]
 [0.36134824 0.63865176]
 [0.18345809 0.81654191]
 [0.07742791 0.92257209]]
